# Dense retrieval & embeddings — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def cos(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))
print('setup ok')

## Learned vectors retrieve by semantic direction
A two-dimensional embedding space shows cosine scoring, unit-normalized dot products, lexical mismatch, exact dense search cost, and the effect of a poor embedding geometry.

In [ ]:
# 1) Dense cosine scores retrieve semantic neighbors
D=np.array([[1.0,0.0],[0.8,0.2],[0.0,1.0],[-0.2,0.9]])
q=np.array([0.9,0.1]); scores=np.array([cos(q,d) for d in D])
plt.figure(figsize=(4,4)); plt.quiver([0]*5,[0]*5,[q[0],*D[:,0]],[q[1],*D[:,1]],angles='xy',scale_units='xy',scale=1); plt.xlim(-.4,1.1); plt.ylim(-.1,1.1); plt.title('query and document embeddings')
assert np.allclose(scores,[0.9938837347,0.9909924304,0.1104315261,-0.1078018268])

In [ ]:
# 2) Unit normalization makes dot product equal cosine
qn=q/np.linalg.norm(q); Dn=D/np.linalg.norm(D,axis=1,keepdims=True); dots=Dn@qn
plt.figure(figsize=(6,3)); plt.bar(['d1','d2','d3','d4'],dots); plt.ylabel('unit dot'); plt.title('normalized dot product = cosine')
assert np.allclose(dots,scores)

In [ ]:
# 3) Dense retrieval can recover a lexical mismatch
lexical=np.array([1,0,0,0])  # only d1 shares exact words in this toy view
plt.figure(figsize=(6,3)); plt.bar(np.arange(4)-.15,lexical,width=.3,label='keyword match'); plt.bar(np.arange(4)+.15,scores,width=.3,label='dense score'); plt.xticks(range(4),['d1','d2','d3','d4']); plt.legend(); plt.title('d2 has no keyword match but high dense score')
assert lexical[1]==0 and scores[1]>0.99

In [ ]:
# 4) Exact dense search costs one comparison per document
Ns=np.array([10,100,1000,10000]); d=2; costs=Ns*d
plt.figure(figsize=(6,3)); plt.loglog(Ns,costs,'-o'); plt.xlabel('documents N'); plt.ylabel('coordinate ops N*d'); plt.title('exact scan grows linearly')
assert costs[-1]/costs[0]==1000

In [ ]:
# 5) A poor embedding direction creates a false neighbor
badD=D.copy(); badD[3]=np.array([0.85,0.15]); bad_scores=np.array([cos(q,d) for d in badD])
plt.figure(figsize=(6,3)); plt.bar(['d1','d2','d3','bad d4'],bad_scores); plt.ylabel('cosine'); plt.title('geometry quality controls retrieval quality')
assert bad_scores[3]>scores[2] and bad_scores[3]>0.99